# Fine-tune PhoBERT for Vietnamese Text Classification

Fine-tune `vinai/phobert-base` trên dữ liệu prelabeled để phân loại: `stress`, `depression`, `political`, `neutral`.

PhoBERT là pre-trained language model tốt nhất cho tiếng Việt, dựa trên kiến trúc RoBERTa.

In [1]:
# Google Colab setup
# On Colab: install dependencies and use GPU runtime (Runtime → Change runtime type → GPU)

import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/social-mental-health-classifier/data"
    IN_COLAB = True
    print("Running on Google Colab")
    !pip install -q transformers datasets accelerate underthesea
except ImportError:
    DATA_DIR = "../data"
    IN_COLAB = False
    print("Running locally")

Running locally


In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Load & Prepare Data

In [ ]:
LABEL2ID = {"stress": 0, "depression": 1, "political": 2, "neutral": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df = pd.read_csv(f"{DATA_DIR}/prelabeled/05_03_2026_gpt4omini.csv")
df = df.dropna(subset=["text_clean", "label"])
df["label_id"] = df["label"].map(LABEL2ID)

X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"].values,
    df["label_id"].values,
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"].values,
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Labels: {LABEL2ID}")

## Tokenizer & Dataset

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = "vinai/phobert-base"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding="max_length",
            max_length=MAX_LEN, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }


train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

## Class Weights (handle imbalance)

In [ ]:
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

for i, w in enumerate(class_weights):
    print(f"  {ID2LABEL[i]}: {w:.3f}")

## Training with Hugging Face Trainer

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID,
).to(device)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}


training_args = TrainingArguments(
    output_dir="./phobert-mental-health",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluation

In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

label_names = [ID2LABEL[i] for i in range(4)]
print(classification_report(y_test, y_pred, target_names=label_names))

# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"PhoBERT — Accuracy: {accuracy_score(y_test, y_pred):.4f}")
plt.tight_layout()
plt.show()

## Training Loss Curve

In [ ]:
log_history = trainer.state.log_history

train_loss = [(log["step"], log["loss"]) for log in log_history if "loss" in log]
eval_acc = [(log["step"], log["eval_accuracy"]) for log in log_history if "eval_accuracy" in log]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
steps, losses = zip(*train_loss)
axes[0].plot(steps, losses)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")

# Eval accuracy
steps, accs = zip(*eval_acc)
axes[1].plot(steps, accs, marker="o")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Eval Accuracy per Epoch")

plt.tight_layout()
plt.show()

## Save Model

In [ ]:
save_dir = "./phobert-mental-health/best"
if IN_COLAB:
    save_dir = f"{DATA_DIR}/../models/phobert-mental-health"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model saved to {save_dir}")